In [5]:
from langgraph.graph import StateGraph,START,END,add_messages
from typing import Optional,TypedDict,Literal,Annotated,operator
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
from pydantic import BaseModel,Field
from langchain_openai import AzureChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
import os

In [6]:
load_dotenv()


AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT_EUS2')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_APIKEY_EUS2')
AZURE_OPENAI_API_VERSION = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
LLM_DEPLOYMENT_NAME = os.getenv('LLM_DEPLOYMENT_NAME', os.getenv('MODEL_NAME'))


model = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=LLM_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
    temperature=0.2,
)


NameError: name 'load_dotenv' is not defined

In [ ]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = model.invoke(messages)
    return {'messages':[response]}


In [ ]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['messages'][-1].content

In [ ]:
thread_id  = '1'

while True:
    user_message = input("User: ")
    print("UUser:", user_message)
    if user_message.strip().lower() in ['exit', 'quit','bye']:
        break
    config = {'configurable': {'thread_id': thread_id}}

    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    print("Chatbot:", response['messages'][-1].content)